<a href="https://colab.research.google.com/github/nikolas-joyce/alpha-momentum-rank/blob/main/notebooks/alpha_momentum_rank.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Alpha: Cross-Sectional Momentum Rank (L3/S3)

> Ranks each ticker by its 1-month / 12-month return relative to the universe. The top 30% by rank (≥70th percentile) enter long; the bottom 30% (≤30th percentile) enter short. This cross-sectional approach is robust to market-wide moves and captures relative strength across asset classes.

## Notebook structure
| Section | Description |
|---------|-------------|
| 0 | Install & imports |
| 1 | Config & data |
| 2 | Helper functions |
| 3 | L3/S3 signal generation |
| 4 | Returns & performance |
| 5 | Per-ticker breakdown |
| 6 | Parameter sensitivity (formation period, rank threshold) |
| 7 | Export signals for td-swarm |

**Run all cells top-to-bottom in a fresh Colab runtime.**


In [ ]:
# Cell 0: Installs & Imports
# ============================================================
!pip install yfinance --quiet

import time
import json
import itertools
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
plt.style.use('seaborn-v0_8-darkgrid')

try:
    import yfinance as yf
except ImportError as e:
    raise ImportError('yfinance missing — restart runtime after install') from e

print('Imports complete')

## Section 1 — Config & Data

In [ ]:
# Cell 1: Config & Data
# ============================================================

TICKERS    = ['SPY','QQQ','IWM','TLT','HYG',
              'GLD','USO','UUP','EEM','VNQ','SLV','XLF','XLE']
START_DATE = '2015-01-01'
COST_BPS   = 5
SETUP_LEN   = 9
CD_LEN      = 13
HOLD_WINDOW = 63
TDST_WINDOW = 9
WF_SPLIT_DATE = '2022-01-01'

print('Fetching OHLCV...')
raw = yf.download(TICKERS, start=START_DATE, auto_adjust=True, progress=False)

assert isinstance(raw.columns, pd.MultiIndex), \
    'Expected MultiIndex columns from yf.download'
assert set(['Close','High','Low']).issubset(
    raw.columns.get_level_values(0)), 'Missing OHLC columns'

close   = raw['Close'].dropna(how='all').ffill(limit=3)
high    = raw['High'].dropna(how='all').ffill(limit=3)
low     = raw['Low'].dropna(how='all').ffill(limit=3)
returns = close.pct_change()
idx     = close.index
high    = high.reindex(idx).ffill(limit=3)
low     = low.reindex(idx).ffill(limit=3)

keep = close.isnull().mean() < 0.20
if not keep.all():
    dropped = close.columns[~keep].tolist()
    print(f'  Dropping tickers with >20% NaN: {dropped}')
    close, high, low, returns = (
        df[keep[keep].index] for df in [close, high, low, returns])
    TICKERS = close.columns.tolist()

print(f'  Shape  : {close.shape}')
print(f'  Range  : {close.index[0].date()} -> {close.index[-1].date()}')
print(f'  Tickers: {TICKERS}')
print('Data ready')

## Section 2 — Helper Functions

In [ ]:
# Cell 2: Helper Functions
# ============================================================
# FIX: compute_atr — removed deprecated groupby(level=1, axis=1).
# FIX: apply_costs — generalised to Series or DataFrame, returns SIGNED P&L.

def compute_atr(high, low, close, period=14):
    """Average True Range — vectorised."""
    prev = close.shift(1)
    tr   = pd.DataFrame(
        np.maximum(
            np.maximum(high.values - low.values,
                       np.abs(high.values - prev.values)),
            np.abs(low.values - prev.values)),
        index=close.index, columns=close.columns)
    return tr.rolling(period, min_periods=period).mean()


def apply_costs(returns, position, cost_bps=5):
    """
    Signed net P&L after transaction costs.
    result = returns * position - |d(position)| * cost
    The sign of position is preserved — this is a SIGNED P&L series.
    Accepts Series or DataFrame; aligns index automatically.
    """
    cost     = cost_bps / 10_000
    position = position.reindex(returns.index).fillna(0)
    turnover = position.diff().abs()
    return returns * position - turnover * cost


def performance_summary(r, name=''):
    """Annualised stats for a daily return Series."""
    r = r.dropna()
    if len(r) < 21:
        return pd.Series(
            dict(zip(['Ann. Return','Ann. Vol','Sharpe','Max DD','Calmar','Hit Rate'],
                     [np.nan]*6)), name=name)
    ann     = 252
    cum     = (1 + r).cumprod()
    ann_ret = (1 + r.mean()) ** ann - 1
    ann_vol = r.std() * np.sqrt(ann)
    sharpe  = ann_ret / ann_vol if ann_vol > 1e-9 else np.nan
    mdd     = (cum / cum.cummax() - 1).min()
    calmar  = ann_ret / abs(mdd) if abs(mdd) > 1e-9 else np.nan
    hits    = (r > 0).mean()
    return pd.Series({
        'Ann. Return': ann_ret, 'Ann. Vol': ann_vol,
        'Sharpe': sharpe, 'Max DD': mdd,
        'Calmar': calmar, 'Hit Rate': hits,
    }, name=name)


def score_variant(stats):
    """Composite swarm score: Sharpe×0.4 + Calmar×0.3 + HitRate×0.3."""
    s = stats.get('Sharpe', np.nan)
    c = stats.get('Calmar', np.nan)
    h = stats.get('Hit Rate', np.nan)
    if any(np.isnan(v) for v in [s, c, h]):
        return -999.0
    return s * 0.40 + c * 0.30 + h * 0.30


print('Helpers loaded')

## Section 3 — Signal Generation

In [ ]:
# Momentum Rank Signal (L3/S3)
# L3: 1-month / 12-month relative momentum rank >= 70th percentile → long
# S3: rank <= 30th percentile → short

ALPHA_LABEL = 'Momentum_Rank'

mom_rank = (close.shift(21) / close.shift(252) - 1).rank(axis=1, pct=True)

v_long  = (mom_rank >= 0.70).astype(int).shift(1).fillna(0)
v_short = (mom_rank <= 0.30).astype(int).shift(1).fillna(0)

print(f'L3 density: {v_long.mean().mean():.1%}')
print(f'S3 density: {v_short.mean().mean():.1%}')


## Section 4 — Returns & Performance

In [ ]:
# Returns & Performance — single-alpha

def compute_alpha_returns(long_sig, short_sig, returns, label, cost_bps=COST_BPS):
    def _side(sig, direction, side_label):
        pos = sig.reindex(returns.index).fillna(0) * direction
        net = apply_costs(returns, pos, cost_bps)
        n   = sig.sum(axis=1).replace(0, float('nan'))
        pnl = net.sum(axis=1)
        return (pnl / n).fillna(0).rename(f'{label}_{side_label}')
    l_ret    = _side(long_sig,  +1, 'Long')
    s_ret    = _side(short_sig, -1, 'Short')
    combined = ((l_ret + s_ret) / 2).rename(label)
    return l_ret, s_ret, combined

a_long_ret, a_short_ret, a_combined = compute_alpha_returns(
    v_long, v_short, returns, ALPHA_LABEL)

print(f'Alpha: {ALPHA_LABEL}')
print(performance_summary(a_long_ret,  name='Long side').round(3).to_string())
print()
print(performance_summary(a_short_ret, name='Short side').round(3).to_string())
print()
print(performance_summary(a_combined,  name='Combined').round(3).to_string())

# Equity curve
cum = (1 + a_combined).cumprod()
fig, ax = plt.subplots(figsize=(14, 5))
cum.plot(ax=ax, title=f'{ALPHA_LABEL} — Equity Curve')
ax.set_ylabel('Cumulative return')
plt.tight_layout()
plt.show()


## Section 5 — Per-Ticker Breakdown

In [ ]:
# Per-Ticker Performance Breakdown

def per_ticker_sharpe_single(long_sig, short_sig, returns, label):
    sharpes = {}
    for ticker in TICKERS:
        lp  = long_sig[ticker].reindex(returns.index).fillna(0)
        sp  = short_sig[ticker].reindex(returns.index).fillna(0)
        ln  = apply_costs(returns[[ticker]], lp.to_frame(), COST_BPS)[ticker]
        sn  = apply_costs(returns[[ticker]], (sp*-1).to_frame(), COST_BPS)[ticker]
        cr  = ((ln + sn)/2).dropna()
        std = cr.std()
        sharpes[ticker] = (cr.mean()/std*np.sqrt(252) if std>1e-9 else float('nan'))
    return pd.Series(sharpes, name=label)

ts = per_ticker_sharpe_single(v_long, v_short, returns, ALPHA_LABEL)
print(f'Per-Ticker Sharpe ({ALPHA_LABEL}):')
print(ts.sort_values(ascending=False).round(3).to_string())

fig, ax = plt.subplots(figsize=(12, 4))
ts.sort_values().plot(kind='barh', ax=ax, title=f'{ALPHA_LABEL} — Per-Ticker Sharpe')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()


## Section 6 — Parameter Sensitivity

In [ ]:
# Parameter Sensitivity — Momentum Rank
# Vary formation period and rank threshold

import itertools
sweep_rows = []
for lookback, threshold in itertools.product([63, 126, 189, 252], [0.65, 0.70, 0.75, 0.80]):
    mr  = (close.shift(21) / close.shift(lookback) - 1).rank(axis=1, pct=True)
    vl  = (mr >= threshold).astype(int).shift(1).fillna(0)
    vs  = (mr <= (1 - threshold)).astype(int).shift(1).fillna(0)
    _, _, comb = compute_alpha_returns(vl, vs, returns, f'lb{lookback}_t{threshold}')
    sweep_rows.append({**performance_summary(comb).to_dict(),
                       'lookback': lookback, 'threshold': threshold})
sweep_df = pd.DataFrame(sweep_rows).sort_values('Sharpe', ascending=False)
print(sweep_df[['lookback','threshold','Sharpe','Ann. Return','Max DD']].round(3).to_string())


## Section 7 — Export Signals for Swarm

In [ ]:
import os, pickle
os.makedirs('exports', exist_ok=True)
with open('exports/alpha_momentum_rank.pkl', 'wb') as f:
    pickle.dump({'long': v_long, 'short': v_short}, f)
print(f'Signals exported -> exports/alpha_momentum_rank.pkl')
